In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Hello world

<details>
<summary><b>Package versions</b></summary>

โค้ดในหน้านี้พัฒนาโดยใช้ requirement ต่อไปนี้
แนะนำให้ใช้เวอร์ชันเหล่านี้หรือใหม่กว่า

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
ตัวอย่างนี้แบ่งเป็นสองส่วน ส่วนแรกจะสร้างโปรแกรมควอนตัมง่าย ๆ แล้วรันบนหน่วยประมวลผลควอนตัม (QPU) เนื่องจากงานวิจัยควอนตัมจริงต้องการโปรแกรมที่แข็งแกร่งกว่านี้มาก ในส่วนที่สอง ([ขยายไปยังจำนวน Qubit ขนาดใหญ่](#scale-to-large-numbers-of-qubits)) จะนำโปรแกรมง่าย ๆ นั้นขยายขึ้นไปสู่ระดับ utility

## ติดตั้งและยืนยันตัวตน
1. ถ้ายังไม่ได้ติดตั้ง Qiskit ดูคำแนะนำได้ที่คู่มือ [Quickstart](/guides/quick-start)

    - ติดตั้ง Qiskit Runtime เพื่อรัน job บนฮาร์ดแวร์ควอนตัม:

        ```bash
        pip install qiskit-ibm-runtime
        ```

    - ตั้งค่า environment เพื่อรัน Jupyter notebook บนเครื่องของตัวเอง:

        ```bash
        pip install jupyter
        ```

2. ตั้งค่าการยืนยันตัวตนเพื่อเข้าถึงฮาร์ดแวร์ควอนตัมผ่าน [Open Plan](/guides/plans-overview#open-plan) ฟรี

    (ถ้าได้รับอีเมลเชิญเข้าร่วมบัญชี ให้ทำตาม [ขั้นตอนสำหรับผู้ใช้ที่ได้รับเชิญ](/guides/cloud-setup-invited) แทน)

    - ไปที่ [IBM Quantum Platform](https://quantum.cloud.ibm.com/) เพื่อเข้าสู่ระบบหรือสร้างบัญชีใหม่
         > **Note:** ถ้าเชื่อมต่อผ่าน proxy server ต้องใช้ Qiskit Runtime v0.44.0 ขึ้นไป
    - สร้าง API key (เรียกอีกชื่อว่า *API token*) บน [dashboard](https://quantum.cloud.ibm.com/) แล้วคัดลอกไปเก็บไว้ในที่ปลอดภัย
    - ไปที่หน้า [Instances](https://quantum.cloud.ibm.com/instances) แล้วหา instance ที่ต้องการใช้ เลื่อนเมาส์ไปที่ CRN แล้วคลิกเพื่อคัดลอก

    - บันทึก credential ไว้บนเครื่องด้วยโค้ดนี้:

        ```python
        from qiskit_ibm_runtime import QiskitRuntimeService

        QiskitRuntimeService.save_account(
        token="<your-api-key>", # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
        instance="<CRN>", # Optional
        )
        ```

3. ตอนนี้ใช้โค้ด Python นี้ได้ทุกครั้งที่ต้องการยืนยันตัวตนกับ Qiskit Runtime Service:
    ```python
        from qiskit_ibm_runtime import QiskitRuntimeService

        # Run every time you need the service
        service = QiskitRuntimeService()
    ```
> **Info:** ถ้าใช้คอมพิวเตอร์สาธารณะหรือ environment ที่ไม่ปลอดภัย ให้ทำตาม [คำแนะนำการยืนยันตัวตนแบบ manual](/guides/cloud-setup-untrusted) แทน เพื่อให้ credential ปลอดภัย
## สร้างและรันโปรแกรมควอนตัมง่าย ๆ
ขั้นตอนสี่ขั้นในการเขียนโปรแกรมควอนตัมด้วย Qiskit patterns คือ:

1.  แมปปัญหาให้อยู่ในรูปแบบที่เป็น native ของควอนตัม

2.  ปรับ Circuit และ operator ให้เหมาะสม

3.  รันโดยใช้ฟังก์ชัน quantum primitive

4.  วิเคราะห์ผลลัพธ์

### ขั้นตอนที่ 1 แมปปัญหาให้อยู่ในรูปแบบที่เป็น native ของควอนตัม
ในโปรแกรมควอนตัม *quantum Circuit* คือรูปแบบ native สำหรับแทนคำสั่งควอนตัม และ *operator* แทน observable ที่ต้องการวัด เมื่อสร้าง Circuit มักจะสร้าง object [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class) ใหม่ แล้วเพิ่มคำสั่งลงไปตามลำดับ
โค้ดต่อไปนี้สร้าง Circuit ที่ผลิต *Bell state* ซึ่งเป็นสถานะที่ Qubit สอง Qubit พันกัน (entangled) กันอย่างสมบูรณ์

> **Note:** Qiskit SDK ใช้การนับบิตแบบ LSb 0 โดยที่หลักที่ $n$ มีค่า $1 \ll n$ หรือ $2^n$ สำหรับรายละเอียดเพิ่มเติม ดูที่หัวข้อ [Bit-ordering in the Qiskit SDK](/guides/bit-ordering)

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorOptions
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from matplotlib import pyplot as plt
# Uncomment the next line if you want to use a simulator:
# from qiskit_ibm_runtime.fake_provider import FakeBelemV2


# Create a new circuit with two qubits
qc = QuantumCircuit(2)

# Add a Hadamard gate to qubit 0
qc.h(0)

# Perform a controlled-X gate on qubit 1, controlled by qubit 0
qc.cx(0, 1)

# Return a drawing of the circuit using MatPlotLib ("mpl").
# These guides are written by using Jupyter notebooks, which
# display the output of the last line of each cell.
# If you're running this in a script, use `print(qc.draw())` to
# print a text drawing.
qc.draw("mpl")

<Image src="../docs/images/guides/hello-world/extracted-outputs/930ca3b6-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/hello-world/extracted-outputs/930ca3b6-0.svg)

ดู [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class) ในเอกสารสำหรับ operation ที่มีทั้งหมด
เมื่อสร้าง quantum Circuit ต้องพิจารณาด้วยว่าต้องการข้อมูลประเภทใดหลังการรัน Qiskit มีสองวิธีในการคืนค่าข้อมูล: รับ probability distribution ของชุด Qubit ที่เลือกวัด หรือรับค่าความคาดหวัง (expectation value) ของ observable เตรียม workload ให้วัด Circuit ด้วยวิธีใดวิธีหนึ่งใน [Qiskit primitives](/guides/get-started-with-primitives) (อธิบายรายละเอียดใน [ขั้นตอนที่ 3](#step-3-execute-using-the-quantum-primitives))

ตัวอย่างนี้วัด expectation value โดยใช้ submodule `qiskit.quantum_info` ซึ่งระบุโดยใช้ operator (วัตถุทางคณิตศาสตร์ที่ใช้แทนการกระทำหรือกระบวนการที่เปลี่ยนสถานะควอนตัม) โค้ดต่อไปนี้สร้าง Pauli operator สองตัว Qubit จำนวนหก ตัว ได้แก่ `IZ`, `IX`, `ZI`, `XI`, `ZZ` และ `XX`

In [2]:
# Set up six different observables.

observables_labels = ["IZ", "IX", "ZI", "XI", "ZZ", "XX"]
observables = [SparsePauliOp(label) for label in observables_labels]

> **Note:** ที่นี่ operator อย่าง `ZZ` เป็นชื่อย่อสำหรับ tensor product $Z\otimes Z$ ซึ่งหมายถึงการวัด Z บน Qubit 1 และ Z บน Qubit 0 พร้อมกัน และได้ข้อมูลเกี่ยวกับความสัมพันธ์ระหว่าง Qubit 1 และ Qubit 0 expectation value แบบนี้มักเขียนเป็น $\langle Z_1 Z_0 \rangle$
> 
> ถ้าสถานะ entangled การวัด $\langle Z_1 Z_0 \rangle$ ควรแตกต่างจากการวัด $\langle I_1 \otimes Z_0 \rangle \langle Z_1 \otimes I_0 \rangle$ สำหรับสถานะ entangled เฉพาะที่สร้างโดย Circuit ที่อธิบายไว้ข้างต้น การวัด $\langle Z_1 Z_0 \rangle$ ควรได้ 1 และการวัด $\langle I_1 \otimes Z_0 \rangle \langle Z_1 \otimes I_0 \rangle$ ควรได้ศูนย์
<span id="optimize"></span>

### ขั้นตอนที่ 2 ปรับ Circuit และ operator ให้เหมาะสม

เมื่อรัน Circuit บนอุปกรณ์ สิ่งสำคัญคือต้องปรับชุดคำสั่งที่ Circuit ประกอบด้วยให้เหมาะสม และลด depth โดยรวม (คร่าว ๆ คือจำนวนคำสั่ง) ของ Circuit ให้น้อยที่สุด เพื่อให้ได้ผลลัพธ์ที่ดีที่สุดโดยการลดผลกระทบจากข้อผิดพลาดและสัญญาณรบกวน นอกจากนี้ คำสั่งของ Circuit ต้องสอดคล้องกับ [Instruction Set Architecture (ISA)](/guides/transpile#instruction-set-architecture) ของ Backend และต้องพิจารณา basis Gate และการเชื่อมต่อ Qubit ของอุปกรณ์

โค้ดต่อไปนี้ instantiate อุปกรณ์จริงเพื่อส่ง job ไป และแปลง Circuit กับ observable ให้ตรงกับ ISA ของ Backend นั้น โดยต้องการให้ [บันทึก credential ไว้แล้ว](/guides/cloud-setup)

In [ ]:
service = QiskitRuntimeService()

backend = service.least_busy(simulator=False, operational=True)

# Convert to an ISA circuit and layout-mapped observables.
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

isa_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/hello-world/extracted-outputs/9a901271-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/hello-world/extracted-outputs/9a901271-0.svg)

### ขั้นตอนที่ 3 รันโดยใช้ quantum primitive
คอมพิวเตอร์ควอนตัมสามารถให้ผลลัพธ์แบบสุ่ม ดังนั้นมักจะเก็บตัวอย่าง output โดยการรัน Circuit หลายครั้ง สามารถประมาณค่าของ observable ได้โดยใช้คลาส `Estimator` `Estimator` เป็นหนึ่งในสอง [primitive](/guides/get-started-with-primitives) อีกตัวคือ `Sampler` ซึ่งใช้รับข้อมูลจากคอมพิวเตอร์ควอนตัม object เหล่านี้มีเมธอด `run()` ที่รันชุด Circuit, observable และ parameter (ถ้ามี) โดยใช้ [primitive unified bloc (PUB)](/guides/primitives#sampler)

In [4]:
# Construct the Estimator instance.

estimator = Estimator(mode=backend)
estimator.options.resilience_level = 1
estimator.options.default_shots = 5000

mapped_observables = [
    observable.apply_layout(isa_circuit.layout) for observable in observables
]

# One pub, with one circuit to run against five different observables.
job = estimator.run([(isa_circuit, mapped_observables)])

# Use the job ID to retrieve your job data later
print(f">>> Job ID: {job.job_id()}")

>>> Job ID: d5k96q4jt3vs73ds5tgg


After a job is submitted, you can wait until either the job is completed within your current python instance, or use the `job_id` to retrieve the data at a later time.  (See the [section on retrieving jobs](/docs/guides/monitor-job#retrieve-job-results-at-a-later-time) for details.)

After the job completes, examine its output through the job's `result()` attribute.

In [5]:
# This is the result of the entire submission.  You submitted one Pub,
# so this contains one inner result (and some metadata of its own).
job_result = job.result()

# This is the result from our single pub, which had six observables,
# so contains information on all six.
pub_result = job.result()[0]

In [6]:
# Check there are six observables.
# If not, edit the comments in the previous cell and update this test.
assert len(pub_result.data.evs) == 6

<Admonition type="note" title="Alternative: run the example using a simulator">
  When you run your quantum program on a real device, your workload must wait in a queue before it runs. To save time, you can instead use the following code to run this small workload on the [`fake_provider`](../api/qiskit-ibm-runtime/fake-provider) with the Qiskit Runtime local testing mode. Note that this is only possible for a small circuit. When you scale up in the next section, you will need to use a real device.

  ```python

  # Use the following code instead if you want to run on a simulator:

  from qiskit_ibm_runtime.fake_provider import FakeBelemV2
  backend = FakeBelemV2()
  estimator = Estimator(backend)

  # Convert to an ISA circuit and layout-mapped observables.

  pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
  isa_circuit = pm.run(qc)
  mapped_observables = [
      observable.apply_layout(isa_circuit.layout) for observable in observables
  ]

  job = estimator.run([(isa_circuit, mapped_observables)])
  result = job.result()

  # This is the result of the entire submission.  You submitted one Pub,
  # so this contains one inner result (and some metadata of its own).

  job_result = job.result()

  # This is the result from our single pub, which had five observables,
  # so contains information on all five.

  pub_result = job.result()[0]
  ```
</Admonition>

### Step 4. Analyze the results

The analyze step is typically where you might post-process your results using, for example, measurement error mitigation or zero noise extrapolation (ZNE). You might feed these results into another workflow for further analysis or prepare a plot of the key values and data. In general, this step is specific to your problem.  For this example, plot each of the expectation values that were measured for our circuit.

The expectation values and standard deviations for the observables you specified to Estimator are accessed through the job result's `PubResult.data.evs` and `PubResult.data.stds` attributes. To obtain the results from Sampler, use the `PubResult.data.meas.get_counts()` function, which will return a `dict` of measurements in the form of bitstrings as keys and counts as their corresponding values. For more information, see [Get started with Sampler.](/docs/guides/get-started-with-primitives#get-started-with-sampler)

In [ ]:
# Plot the result

values = pub_result.data.evs

errors = pub_result.data.stds

# plotting graph
plt.plot(observables_labels, values, "-o")
plt.xlabel("Observables")
plt.ylabel("Values")
plt.show()

<Image src="../docs/images/guides/hello-world/extracted-outputs/87143fcc-0.svg" alt="Output of the previous code cell" />

> **Note:** เมื่อรันโปรแกรมควอนตัมบนอุปกรณ์จริง workload ต้องรอในคิวก่อน เพื่อประหยัดเวลา ใช้โค้ดต่อไปนี้แทนเพื่อรัน workload ขนาดเล็กนี้บน [`fake_provider`](../api/qiskit-ibm-runtime/fake-provider) ด้วย Qiskit Runtime local testing mode โปรดทราบว่าทำได้เฉพาะ Circuit ขนาดเล็กเท่านั้น เมื่อขยายขนาดในส่วนถัดไป ต้องใช้อุปกรณ์จริง
> 
>   ```python
> 
>   # Use the following code instead if you want to run on a simulator:
> 
>   from qiskit_ibm_runtime.fake_provider import FakeBelemV2
>   backend = FakeBelemV2()
>   estimator = Estimator(backend)
> 
>   # Convert to an ISA circuit and layout-mapped observables.
> 
>   pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
>   isa_circuit = pm.run(qc)
>   mapped_observables = [
>       observable.apply_layout(isa_circuit.layout) for observable in observables
>   ]
> 
>   job = estimator.run([(isa_circuit, mapped_observables)])
>   result = job.result()
> 
>   # This is the result of the entire submission.  You submitted one Pub,
>   # so this contains one inner result (and some metadata of its own).
> 
>   job_result = job.result()
> 
>   # This is the result from our single pub, which had five observables,
>   # so contains information on all five.
> 
>   pub_result = job.result()[0]
>   ```
### ขั้นตอนที่ 4 วิเคราะห์ผลลัพธ์
ขั้นตอนการวิเคราะห์มักเป็นขั้นตอนที่อาจทำการ post-process ผลลัพธ์ เช่น การบรรเทาข้อผิดพลาดจากการวัด (measurement error mitigation) หรือ zero noise extrapolation (ZNE) อาจนำผลลัพธ์เหล่านี้ป้อนเข้า workflow อื่นเพื่อการวิเคราะห์เพิ่มเติม หรือเตรียมกราฟของค่าและข้อมูลสำคัญ โดยทั่วไปขั้นตอนนี้เฉพาะเจาะจงกับปัญหาของตัวเอง สำหรับตัวอย่างนี้ ให้พล็อต expectation value แต่ละค่าที่วัดได้สำหรับ Circuit

expectation value และ standard deviation สำหรับ observable ที่ระบุให้ Estimator เข้าถึงได้ผ่าน attribute `PubResult.data.evs` และ `PubResult.data.stds` ของผลลัพธ์ job หากต้องการผลลัพธ์จาก Sampler ให้ใช้ฟังก์ชัน `PubResult.data.meas.get_counts()` ซึ่งจะคืน `dict` ของการวัดในรูปแบบ bitstring เป็น key และจำนวนนับเป็นค่าที่สอดคล้องกัน สำหรับข้อมูลเพิ่มเติม ดู [เริ่มต้นใช้งาน Sampler](/guides/get-started-with-primitives#get-started-with-sampler)

In [8]:
# Make sure the results follow the claim from the previous markdown cell.
# This can happen when the device occasionally behaves strangely. If this cell
# fails, you may just need to run the notebook again.

_results = {obs: val for obs, val in zip(observables_labels, values)}
for _label in ["IZ", "IX", "ZI", "XI"]:
    assert abs(_results[_label]) < 0.2
for _label in ["XX", "ZZ"]:
    assert _results[_label] > 0.8

![Output of the previous code cell](../docs/images/guides/hello-world/extracted-outputs/87143fcc-0.svg)

สังเกตว่าสำหรับ Qubit 0 และ 1 expectation value อิสระของทั้ง X และ Z เป็น 0 ในขณะที่ความสัมพันธ์ (`XX` และ `ZZ`) เป็น 1 นี่คือลักษณะเด่นของการพันกันควอนตัม (quantum entanglement)

In [ ]:
def get_qc_for_n_qubit_GHZ_state(n: int) -> QuantumCircuit:
    """This function will create a qiskit.QuantumCircuit (qc) for an n-qubit GHZ state.

    Args:
        n (int): Number of qubits in the n-qubit GHZ state

    Returns:
        QuantumCircuit: Quantum circuit that generate the n-qubit GHZ state, assuming all qubits start in the 0 state
    """
    if isinstance(n, int) and n >= 2:
        qc = QuantumCircuit(n)
        qc.h(0)
        for i in range(n - 1):
            qc.cx(i, i + 1)
    else:
        raise Exception("n is not a valid input")
    return qc


# Create a new circuit with two qubits (first argument) and two classical
# bits (second argument)
n = 100
qc = get_qc_for_n_qubit_GHZ_state(n)

## ขยายขนาดไปสู่ Qubit จำนวนมาก
ในการคำนวณเชิงควอนตัม งานในระดับ utility-scale เป็นสิ่งสำคัญมากสำหรับความก้าวหน้าในสาขานี้ งานดังกล่าวต้องทำการคำนวณในขนาดที่ใหญ่กว่ามาก โดยทำงานกับ Circuit ที่อาจใช้ Qubit มากกว่า 100 ตัวและ Gate มากกว่า 1000 ตัว ตัวอย่างนี้แสดงให้เห็นว่าเราสามารถทำงานในระดับ utility-scale บน IBM&reg; QPU ได้อย่างไร โดยการสร้างและวิเคราะห์ GHZ state ขนาด 100 Qubit ตัวอย่างนี้ใช้ workflow แบบ Qiskit patterns และจบด้วยการวัดค่า expectation value $\langle Z_0 Z_i \rangle$ สำหรับแต่ละ Qubit

### ขั้นตอนที่ 1 กำหนดปัญหา
เขียนฟังก์ชันที่คืนค่า `QuantumCircuit` ซึ่งเตรียม GHZ state แบบ $n$-Qubit (โดยพื้นฐานคือ Bell state ที่ขยายออกไป) จากนั้นใช้ฟังก์ชันนั้นเพื่อเตรียม GHZ state แบบ 100 Qubit และรวบรวม observable ที่ต้องการวัด

In [ ]:
# ZZII...II, ZIZI...II, ... , ZIII...IZ
operator_strings = [
    "Z" + "I" * i + "Z" + "I" * (n - 2 - i) for i in range(n - 1)
]
print(operator_strings)
print(len(operator_strings))

operators = [SparsePauliOp(operator) for operator in operator_strings]

['ZZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII

ต่อไป ทำการ map ไปยัง operator ที่ต้องการ ตัวอย่างนี้ใช้ `ZZ` operators ระหว่าง Qubit เพื่อตรวจสอบพฤติกรรมเมื่อ Qubit อยู่ห่างออกไปเรื่อยๆ ค่า expectation value ที่ไม่แม่นยำมากขึ้น (บิดเบือน) ระหว่าง Qubit ที่อยู่ห่างไกลจะเผยให้เห็นระดับของ noise ที่มีอยู่

In [ ]:
service = QiskitRuntimeService()

backend = service.least_busy(
    simulator=False, operational=True, min_num_qubits=100
)
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)

isa_circuit = pm.run(qc)
isa_operators_list = [op.apply_layout(isa_circuit.layout) for op in operators]

### Step 3. Execute on hardware

Submit the job and enable error suppression by using a technique to reduce errors called [dynamical decoupling.](../api/qiskit-ibm-runtime/options-dynamical-decoupling-options) The resilience level specifies how much resilience to build against errors. Higher levels generate more accurate results, at the expense of longer processing times.  For further explanation of the options set in the following code, see [Configure error mitigation for Qiskit Runtime.](/docs/guides/configure-error-mitigation)

In [ ]:
options = EstimatorOptions()
options.resilience_level = 1
options.dynamical_decoupling.enable = True
options.dynamical_decoupling.sequence_type = "XY4"

# Create an Estimator object
estimator = Estimator(backend, options=options)

In [13]:
# Submit the circuit to Estimator
job = estimator.run([(isa_circuit, isa_operators_list)])
job_id = job.job_id()
print(job_id)

d5k9mmqvcahs73a1ni3g


### ขั้นตอนที่ 3 รันบน hardware
ส่ง job และเปิดใช้ error suppression โดยใช้เทคนิคเพื่อลด error ที่เรียกว่า [dynamical decoupling](../api/qiskit-ibm-runtime/options-dynamical-decoupling-options) ระดับ resilience ระบุว่าจะสร้างความทนทานต่อ error มากแค่ไหน ระดับที่สูงขึ้นจะให้ผลลัพธ์ที่แม่นยำมากขึ้น แต่แลกกับเวลาประมวลผลที่นานขึ้น สำหรับคำอธิบายเพิ่มเติมของ options ที่ตั้งค่าในโค้ดต่อไปนี้ ดูที่ [Configure error mitigation for Qiskit Runtime](/guides/configure-error-mitigation)

In [ ]:
# data
data = list(range(1, len(operators) + 1))  # Distance between the Z operators
result = job.result()[0]
values = result.data.evs  # Expectation value at each Z operator.
values = [
    v / values[0] for v in values
]  # Normalize the expectation values to evaluate how they decay with distance.

# plotting graph
plt.plot(data, values, marker="o", label="100-qubit GHZ state")
plt.xlabel("Distance between qubits $i$")
plt.ylabel(r"$\langle Z_i Z_0 \rangle / \langle Z_1 Z_0 \rangle $")
plt.legend()
plt.show()

<Image src="../docs/images/guides/hello-world/extracted-outputs/de91ebd0-0.svg" alt="Output of the previous code cell" />

The previous plot shows that as the distance between qubits increases, the signal decays because of the presence of noise.

## Next steps

<Admonition type="tip" title="Recommendations">
  -   Try one of these tutorials:
      - [Ground-state energy estimation of the Heisenberg chain with VQE](/docs/tutorials/spin-chain-vqe)
      - Solve optimization problems using [QAOA](/docs/tutorials/quantum-approximate-optimization-algorithm)
      - Train [quantum kernel](/docs/tutorials/quantum-kernel-training) models for machine learning tasks
  - Find detailed installation instructions in the [Install Qiskit](/docs/guides/install-qiskit) guide.
  - If you prefer not to install Qiskit locally, read about options to use Qiskit in an [online development environment.](/docs/guides/online-lab-environments)
  - To save multiple account credentials or to specify other account options, see detailed instructions in the [Save your login credentials](/docs/guides/save-credentials#save-your-access-credentials) guide.
</Admonition>